# Lab 1 - Profiling Local LLMs
**Asliddin Tirkashev - RNU, Group 4230Z**

**CPU track, local CPU, Apple M4 Pro.** I compare three precisions of Llama3.2 1B and two sizes of Qwen3.
I use the official CPU reductions: three timed runs and one versus two parallel requests.
Model calls use num_gpu=0. Measurements come from the declared runtime, not hardware estimates.
The notebook keeps original Ollama response timing fields and the measured model sizes for reproducibility.


## Step 0 — Setup (run every session, ≈ 5 min)
Run the cells of Step 0 **in order**. You do not need to change anything except the settings cell.
If Colab disconnects, run Step 0 again; your finished results are skipped automatically.

### 0.1 Dependencies
In Colab this cell installs the dependencies. My local run uses the prepared environment.

In [1]:
import importlib.util, sys, subprocess
try:
    IN_COLAB=importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB=False
if IN_COLAB:
    subprocess.check_call([sys.executable,'-m','pip','-q','install','requests==2.32.5','pandas==2.2.3','pydantic==2.14.1','langchain-ollama==1.1.0','langchain-core==1.7.4'])
import requests, pandas as pd
import importlib.metadata
print('Python',sys.version.split()[0],'requests',requests.__version__,'pandas',pd.__version__)
print('langchain-ollama',importlib.metadata.version('langchain-ollama'),'langchain-core',importlib.metadata.version('langchain-core'),'pydantic',importlib.metadata.version('pydantic'))


Python 3.12.14 requests 2.32.5 pandas 2.2.3
langchain-ollama 1.1.0 langchain-core 1.7.4 pydantic 2.14.1


### 0.2 Settings
GPU track uses context4096, maximum256 generated tokens and four parallel slots. CPU track uses context2048, maximum200 tokens and two slots. Both use temperature0 and seed42. No other user Ollama server is restarted.

In [2]:
import os, pathlib, platform, shutil, time, json
TRACK=os.environ.get('GENAI_TRACK','CPU').upper()
os.environ['GENAI_TRACK']=TRACK
if IN_COLAB:os.environ['GENAI_IN_COLAB']='1'
USE_DRIVE=False
PRACTICE='L1'
HERE=pathlib.Path.cwd()
DATA=pathlib.Path(os.environ.get('GENAI_DATA',str(HERE.parent/'data/chaoscorp')))
RESULTS=pathlib.Path(os.environ.get('GENAI_L1_RESULTS',str(HERE/'results')))
OLLAMA_URL=os.environ.get('OLLAMA_URL','http://localhost:11434' if IN_COLAB else 'http://localhost:11435')
print('Track:',TRACK,'Runtime:',platform.platform())


Track: CPU Runtime: macOS-26.5.1-arm64-arm-64bit


### 0.3 Runtime check
GPU track records NVIDIA device and memory. CPU track marks NVIDIA VRAM as not applicable. Ollama reports actual model memory and GPU memory separately.

In [3]:
if shutil.which('nvidia-smi'):
    info=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True)
    print(info.stdout.strip() or info.stderr.strip())
else:
    assert TRACK=='CPU', 'GPU track requires an NVIDIA GPU runtime.'
    print('NVIDIA GPU/VRAM: not applicable. CPU track selected.')


NVIDIA GPU/VRAM: not applicable. CPU track selected.


### 0.4 Upload the data pack
A **"Choose Files"** button appears below the cell: select `chaoscorp_data_pack_v2.zip` (from Moodle). With `USE_DRIVE = True` a copy is kept on Drive and you will not be asked again.

In [4]:
import zipfile
if not DATA.exists():
    candidates=[HERE/'data/chaoscorp',HERE.parent/'data/chaoscorp',HERE/'chaoscorp',pathlib.Path('/content/chaoscorp')]
    DATA=next((p for p in candidates if p.exists()),DATA)
if not DATA.exists() and IN_COLAB:
    from google.colab import files
    uploaded=files.upload()
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        assert 'chaoscorp/README.md' in z.namelist()
        z.extractall('/content')
    DATA=pathlib.Path('/content/chaoscorp')
assert (DATA/'incidents_original10.json').exists(), f'Data missing: {DATA}; set GENAI_DATA.'
RESULTS.mkdir(parents=True,exist_ok=True)
print('Data:',DATA,'Results:',RESULTS)


Data: ../data/chaoscorp Results: results


### 0.5 Dedicated Ollama server
The local course server is already started separately. In Colab the cell starts a service configured for four GPU-track requests or two CPU-track requests.

In [5]:
if IN_COLAB:
    if not shutil.which('ollama'):
        subprocess.run(['apt-get','-qq','install','-y','zstd','pciutils'],check=True)
        subprocess.run('curl -fsSL https://ollama.com/install.sh | sh',shell=True,check=True)
    ollama=subprocess.Popen(['ollama','serve'],env={**os.environ,'OLLAMA_NUM_PARALLEL':'4' if TRACK=='GPU' else '2'},stdout=open('/content/ollama.log','w'),stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            response=requests.get(OLLAMA_URL+'/api/version',timeout=2);response.raise_for_status();break
        except requests.RequestException:time.sleep(1)
response=requests.get(OLLAMA_URL+'/api/version',timeout=10);response.raise_for_status()
print(response.json())


{'version': '0.17.4'}


### 0.6 Helper functions (just run this cell)
You will use these helpers in every step. Read the one-line descriptions so you know what they do.

In [6]:
"""Lab 1 GPU/CPU-track profiling. Real Ollama responses, saved incrementally."""
from __future__ import annotations
import argparse
import hashlib
import json
import os
import platform
import shutil
import statistics
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path
import requests
import pandas as pd

HERE=Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
DATA=globals().get('DATA',Path(os.environ.get('GENAI_DATA',str(HERE.parent/'data/chaoscorp'))))
RESULTS=globals().get('RESULTS',Path(os.environ.get('GENAI_L1_RESULTS',str(HERE/'results'))))
OLLAMA_URL=globals().get('OLLAMA_URL',os.environ.get('OLLAMA_URL','http://localhost:11435')).rstrip('/')
TRACK=os.environ.get('GENAI_TRACK','CPU').upper()
if TRACK not in ('CPU','GPU'):raise ValueError('GENAI_TRACK must be CPU or GPU')
IN_COLAB=bool(os.environ.get('COLAB_RELEASE_TAG') or os.environ.get('COLAB_GPU') or os.environ.get('GENAI_IN_COLAB'))
RUNTIME=os.environ.get('GENAI_RUNTIME',('Google Colab ' if IN_COLAB else 'local ')+TRACK)
if TRACK=='GPU':
    EXP_A=['llama3.2:3b-instruct-q4_K_M','llama3.2:3b-instruct-q8_0','llama3.2:3b-instruct-fp16']
    EXP_B=['qwen3:1.7b-q4_K_M','qwen3:4b-q4_K_M','qwen3:8b-q4_K_M']
else:
    EXP_A=['llama3.2:1b-instruct-q4_K_M','llama3.2:1b-instruct-q8_0','llama3.2:1b-instruct-fp16']
    EXP_B=['qwen3:0.6b-q4_K_M','qwen3:1.7b-q4_K_M']
N_RUNS=5 if TRACK=='GPU' else 3
N_PARALLEL=4 if TRACK=='GPU' else 2
OPTIONS={'temperature':0,'seed':42,'num_ctx':4096 if TRACK=='GPU' else 2048,
         'num_predict':256 if TRACK=='GPU' else 200,'num_gpu':99 if TRACK=='GPU' else 0}
PROMPT='Explain to a new on-call engineer, in at most 120 words, why a read replica can show higher latency than the primary database.'

def read_jsonl(path):
    p=Path(path)
    return [json.loads(line) for line in p.read_text().splitlines() if line.strip()] if p.exists() else []

def append_jsonl(path,row):
    p=Path(path);p.parent.mkdir(parents=True,exist_ok=True)
    with p.open('a',encoding='utf-8') as f:
        f.write(json.dumps(row,ensure_ascii=False,allow_nan=False)+'\n');f.flush();os.fsync(f.fileno())

def decode_tps(resp):
    return resp['eval_count']/(resp['eval_duration']/1e9) if resp.get('eval_duration',0)>0 else 0.0

def prompt_tps(resp):
    return resp['prompt_eval_count']/(resp['prompt_eval_duration']/1e9) if resp.get('prompt_eval_duration',0)>0 else 0.0

def api(path,body=None,timeout=900):
    r=requests.get(OLLAMA_URL+path,timeout=timeout) if body is None else requests.post(OLLAMA_URL+path,json=body,timeout=timeout)
    r.raise_for_status();return r.json()

def loaded_models():
    return api('/api/ps',timeout=10).get('models',[])

def unload(model):
    return api('/api/generate',{'model':model,'keep_alive':0},timeout=90)

def unload_all():
    for row in loaded_models():unload(row['name'])
    for _ in range(60):
        if not loaded_models():return
        time.sleep(.2)
    raise RuntimeError('A model remained loaded before the next benchmark.')

def pull(model):
    return api('/api/pull',{'model':model,'stream':False},timeout=3600)

def chat(model,messages,phase,options=None,case=None):
    body={'model':model,'messages':messages,'stream':False,'keep_alive':'10m','options':{**OPTIONS,**(options or {})}}
    if model.startswith('qwen3'):body['think']=False
    started=time.perf_counter();response=api('/api/chat',body)
    elapsed=time.perf_counter()-started
    append_jsonl(RESULTS/'responses.jsonl',{'timestamp_utc':datetime.now(timezone.utc).isoformat(),'model':model,'phase':phase,'case':case,'options':body['options'],'wall_s':elapsed,'response':response})
    return response

def run(model,prompt,phase='timed',case=None):
    """Standard prompt call with fixed settings; returns Ollama's full response."""
    return chat(model,[{'role':'user','content':prompt}],phase,case=case)

def memory(model):
    row=next((r for r in loaded_models() if r['name']==model or r.get('model')==model),None)
    if row is None:raise RuntimeError(f'Loaded model missing from /api/ps: {model}')
    size=row.get('size',0);vram=row.get('size_vram',0)
    if TRACK=='CPU' and vram:raise RuntimeError(f'Declared CPU run used GPU memory ({vram} bytes).')
    share=vram/size if size else 0
    processor='100% CPU' if share==0 else ('100% GPU' if share>=.99 else f'{100*share:.1f}% GPU / {100*(1-share):.1f}% CPU')
    return {'size_gb':size/1e9,'vram_gb':vram/1e9,'gpu_share':vram/size if size else 0,
            'processor':processor,'gpu_comparable':TRACK!='GPU' or share>=.99,'ollama_ps':row}

def nvidia_memory_gb():
    if TRACK!='GPU' or not shutil.which('nvidia-smi'):return None
    result=subprocess.run(['nvidia-smi','--query-gpu=memory.used','--format=csv,noheader,nounits'],capture_output=True,text=True,check=True)
    return float(result.stdout.splitlines()[0].strip())*1024**2/1e9

def mean_sd(values):
    return statistics.mean(values),statistics.pstdev(values)

def experiment_identity():
    """Inputs which must match before reusing measurements."""
    return {'version':1,'track':TRACK,'options':OPTIONS,'standard_prompt':PROMPT,
            'timed_runs':N_RUNS,'parallel_requests':[1,N_PARALLEL],
            'experiment_a':EXP_A,'experiment_b':EXP_B,
            'incidents_sha256':hashlib.sha256((DATA/'incidents_original10.json').read_bytes()).hexdigest(),
            'severity_policy_sha256':hashlib.sha256((DATA/'policies/policy_severity.md').read_bytes()).hexdigest()}

def ensure_cache_identity():
    identity=experiment_identity()
    path=RESULTS/'experiment_manifest.json'
    if path.exists():
        if json.loads(path.read_text())['identity']!=identity:
            raise RuntimeError('Cached experiment configuration/identity differs; use a new results folder.')
    else:
        if any(read_jsonl(RESULTS/name) for name in ['profiles.jsonl','parallel.jsonl','responses.jsonl']):
            raise RuntimeError('Legacy cached measurements have no identity manifest; audit them or use a new results folder.')
        RESULTS.mkdir(parents=True,exist_ok=True)
        path.write_text(json.dumps({'identity':identity,'established_utc':datetime.now(timezone.utc).isoformat(),
                                    'established_by':'before model calls'},indent=2))
    return identity

def severity_accuracy(model):
    incidents=json.loads((DATA/'incidents_original10.json').read_text())
    guide=(DATA/'policies/policy_severity.md').read_text()
    predictions=[]
    for inc in incidents:
        r=chat(model,[{'role':'system','content':'You classify incident severity using this guide:\n'+guide},
                      {'role':'user','content':inc['raw_log']+'\n\nAnswer with exactly one word: High, Medium or Low.'}],
               'severity',options={'num_predict':10},case=inc['incident_key'])
        answer=r['message']['content'].strip()
        predicted=next((s for s in ('High','Medium','Low') if answer.casefold()==s.casefold()),None)
        predictions.append({'incident_key':inc['incident_key'],'raw':answer,'predicted':predicted,'gold':inc['gold']['severity'],'correct':predicted==inc['gold']['severity']})
    return predictions

def profile_model(model,experiment):
    ensure_cache_identity()
    path=RESULTS/'profiles.jsonl'
    old=next((r for r in read_jsonl(path) if r['model']==model and r['experiment']==experiment),None)
    if old is not None:
        if old.get('options')!=OPTIONS:raise RuntimeError('Cache options differ; use a new result folder.')
        return old
    unload_all()
    before=nvidia_memory_gb()
    load=chat(model,[{'role':'user','content':'Hi'}],'load',{'num_predict':1})
    after=nvidia_memory_gb()
    mem=memory(model)
    run(model,PROMPT,'warmup')
    runs=[run(model,PROMPT,'timed',case=i+1) for i in range(N_RUNS)]
    predictions=severity_accuracy(model)
    row={'model':model,'experiment':experiment,'track':TRACK,'options':OPTIONS,'runs':N_RUNS,
         **mem,'load_s':load['load_duration']/1e9,'severity_correct_of_10':sum(p['correct'] for p in predictions),
         'nvidia_memory_before_gb':before,'nvidia_memory_after_gb':after,
         'nvidia_vram_delta_gb':after-before if before is not None and after is not None else None,
         'severity_predictions':predictions,'raw_timed_responses':runs}
    for metric,values in {'decode_tps':[decode_tps(r) for r in runs],
                          'prompt_tps':[prompt_tps(r) for r in runs],
                          'warm_ttft_s':[r['prompt_eval_duration']/1e9 for r in runs]}.items():
        row[metric+'_mean'],row[metric+'_sd']=mean_sd(values)
    append_jsonl(path,row)
    unload(model)
    print(model,'severity',row['severity_correct_of_10'],'/10; decode',round(row['decode_tps_mean'],2),flush=True)
    return row

def choose_model(rows):
    candidates=[r for r in rows if r['experiment']=='B' and r.get('gpu_comparable',True)]
    if not candidates:raise RuntimeError('No comparable Qwen model measurements are available.')
    # Accuracy first, then measured speed and smaller memory on a tie.
    return max(candidates,key=lambda r:(r['severity_correct_of_10'],r['decode_tps_mean'],-r['size_gb']))

def parallel_test(model,n):
    ensure_cache_identity()
    old=next((r for r in read_jsonl(RESULTS/'parallel.jsonl') if r['model']==model and r['parallel_requests']==n),None)
    if old is not None:return old
    chat(model,[{'role':'user','content':'Hi'}],'parallel_load',{'num_predict':1})
    memory(model)
    started=time.perf_counter()
    with ThreadPoolExecutor(max_workers=n) as pool:
        responses=list(pool.map(lambda i:chat(model,[{'role':'user','content':PROMPT}],f'parallel_{n}',case=i+1),range(n)))
    wall=time.perf_counter()-started
    speeds=[decode_tps(r) for r in responses]
    row={'model':model,'parallel_requests':n,'per_request_tps':speeds,'per_request_tps_mean':statistics.mean(speeds),
         'aggregate_tps':sum(r['eval_count'] for r in responses)/wall,'wall_s':wall,
         'tokens_total':sum(r['eval_count'] for r in responses),'memory':memory(model),'nvidia_memory_gb':nvidia_memory_gb(),'responses':responses}
    append_jsonl(RESULTS/'parallel.jsonl',row)
    return row

def save_summary():
    rows=read_jsonl(RESULTS/'profiles.jsonl')
    flat=[{k:v for k,v in r.items() if k not in ('options','severity_predictions','raw_timed_responses','ollama_ps')} for r in rows]
    pd.DataFrame(flat).to_csv(RESULTS/'profiles.csv',index=False)
    if len(rows)!=len(EXP_A+EXP_B):raise RuntimeError('All model measurements for this track are needed.')
    choice=choose_model(rows)
    parallel=[r for r in read_jsonl(RESULTS/'parallel.jsonl') if r['model']==choice['model']]
    if len(parallel)<2:raise RuntimeError('Both parallel tests are needed.')
    a4=next(r for r in rows if r['model']==EXP_A[0]);a16=next(r for r in rows if r['model']==EXP_A[2])
    ratio=a16['size_gb']/a4['size_gb']
    faster=a4['decode_tps_mean']/a16['decode_tps_mean']
    small=next(r for r in rows if r['model']==EXP_B[0]);large=next(r for r in rows if r['model']==EXP_B[-1])
    pair=next(r for r in parallel if r['parallel_requests']==N_PARALLEL)
    answer_tokens=200
    capacity=pair['aggregate_tps']*20/answer_tokens
    observed_mean_tokens=pair['tokens_total']/pair['parallel_requests']
    answers=[
      f"I measured {a4['size_gb']:.2f} GB for q4_K_M and {a16['size_gb']:.2f} GB for fp16, a {ratio:.2f}-fold memory ratio. Their decode speeds were {a4['decode_tps_mean']:.2f} and {a16['decode_tps_mean']:.2f} tokens/s, respectively ({faster:.2f}x). Severity accuracy was {a4['severity_correct_of_10']}/10 versus {a16['severity_correct_of_10']}/10. I evaluate this trade-off for these logs rather than assuming lower precision always preserves quality.",
      f"In the Qwen3 size comparison, {small['model']} scored {small['severity_correct_of_10']}/10 at {small['decode_tps_mean']:.2f} tokens/s, while {large['model']} scored {large['severity_correct_of_10']}/10 at {large['decode_tps_mean']:.2f} tokens/s. I choose {choice['model']} by highest observed accuracy, then speed and memory efficiency on a tie. The ten-incident sample limits the strength of this comparison.",
      f"The measured {pair['parallel_requests']}-request batch produced {pair['tokens_total']} tokens total, averaging {observed_mean_tokens:.1f} tokens per reply, in {pair['wall_s']:.2f} seconds. Aggregate throughput was {pair['aggregate_tps']:.2f} tokens/s. For a hypothetical 200-token reply, a throughput-only projection gives {pair['aggregate_tps']:.2f} x 20 / 200 = {capacity:.1f} users sending one request every 20 seconds, before safety headroom. The observed reply lengths do not verify a five-second target for 200-token replies. I would test that response length and a realistic arrival pattern before claiming this capacity.",
      f"I select {choice['model']} for later work: it scored {choice['severity_correct_of_10']}/10, decoded at {choice['decode_tps_mean']:.2f} tokens/s and occupied {choice['size_gb']:.2f} GB according to Ollama, including {choice['vram_gb']:.2f} GB of model GPU memory. This is a measured {RUNTIME} result. I keep the model tag and settings fixed in the next tasks."
    ]
    card={'model':choice['model'],'track':TRACK,'runtime':RUNTIME,'decode_tps':choice['decode_tps_mean'],
          'size_gb':choice['size_gb'],'vram_gb':choice['vram_gb'],'severity_correct_of_10':choice['severity_correct_of_10'],
          'reason_for_choice':f"I selected highest measured severity accuracy among Qwen size candidates, breaking ties by measured speed and memory efficiency; accuracy {choice['severity_correct_of_10']}/10, speed {choice['decode_tps_mean']:.2f} tok/s, memory {choice['size_gb']:.2f} GB."}
    (RESULTS/'analysis_answers.json').write_text(json.dumps(answers,indent=2))
    (RESULTS/'model_card.json').write_text(json.dumps(card,indent=2))
    return card

def langchain_call(model):
    from langchain_ollama import ChatOllama
    llm=ChatOllama(model=model,base_url=OLLAMA_URL,temperature=0,seed=42,num_ctx=OPTIONS['num_ctx'],num_predict=OPTIONS['num_predict'],num_gpu=OPTIONS['num_gpu'],
                   **({'reasoning':False} if model.startswith('qwen3') else {}))
    started=time.perf_counter();response=llm.invoke(PROMPT)
    result={'model':model,'wall_s':time.perf_counter()-started,'content':response.content,
            'response_metadata':response.response_metadata,'usage_metadata':response.usage_metadata}
    (RESULTS/'langchain_response.json').write_text(json.dumps(result,ensure_ascii=False,indent=2))
    return result

def main():
    global OLLAMA_URL
    parser=argparse.ArgumentParser();parser.add_argument('--url');parser.add_argument('--pull',action='store_true');parser.add_argument('--profile-only',action='store_true')
    args=parser.parse_args()
    if args.url:OLLAMA_URL=args.url.rstrip('/')
    RESULTS.mkdir(parents=True,exist_ok=True)
    ensure_cache_identity()
    env={'track':TRACK,'runtime':RUNTIME,'platform':platform.platform(),'ollama':api('/api/version'),
         'url':OLLAMA_URL,'options':OPTIONS,'parallel_server_setting':f'OLLAMA_NUM_PARALLEL={N_PARALLEL}; dedicated course server',
         'created_utc':datetime.now(timezone.utc).isoformat()}
    if TRACK=='GPU' and shutil.which('nvidia-smi'):
        env['nvidia_device']=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],capture_output=True,text=True,check=True).stdout.strip()
    runtime_path=RESULTS/'runtime.json'
    if not runtime_path.exists():
        runtime_path.write_text(json.dumps(env,indent=2))
    rows=[]
    for experiment,models in [('A',EXP_A),('B',EXP_B)]:
        for model in models:
            if args.pull:
                print('Preparing model:', model, flush=True)
                pull(model)
            rows.append(profile_model(model,experiment))
    if args.profile_only:return
    choice=choose_model(rows)
    unload_all()
    for n in [1,N_PARALLEL]:print(parallel_test(choice['model'],n),flush=True)
    card=save_summary()
    if not (RESULTS/'langchain_response.json').exists():print(langchain_call(card['model']),flush=True)
    unload_all();print('Lab1 complete:',RESULTS,flush=True)


print('Profiling helpers loaded; every real model response is retained in responses.jsonl.')


Profiling helpers loaded; every real model response is retained in responses.jsonl.


The helpers save raw response fields, use exact severity-only answers, and exclude warm-up calls from the timed averages. They unload the previous model before starting a new profile.

## Step 1 — Reading Ollama's speed numbers
Every Ollama answer contains timing fields (durations are in **nanoseconds**):

| Field | Meaning |
|---|---|
| `eval_count` | number of tokens the model **generated** |
| `eval_duration` | time spent generating them |
| `prompt_eval_count` | number of tokens in the **prompt** |
| `prompt_eval_duration` | time spent reading the prompt |
| `load_duration` | time to load the model into memory |

We use these instead of a stopwatch: they separate *reading the prompt* from *writing the answer*.

### ✏️ YOUR TURN 1 — decode speed and prompt speed
Complete the two functions. Hint: tokens ÷ seconds, and seconds = nanoseconds / 1e9.

In [7]:
def decode_tps(resp):
    return resp['eval_count']/(resp['eval_duration']/1e9) if resp.get('eval_duration',0)>0 else 0.0

def prompt_tps(resp):
    return resp['prompt_eval_count']/(resp['prompt_eval_duration']/1e9) if resp.get('prompt_eval_duration',0)>0 else 0.0


In [8]:
# ✅ Check (do not edit)
fake = {"eval_count": 100, "eval_duration": 2_000_000_000, "prompt_eval_count": 300, "prompt_eval_duration": 500_000_000}
assert decode_tps(fake) is not None and abs(decode_tps(fake) - 50) < 1e-6, "decode_tps should be 50 for this example"
assert prompt_tps(fake) is not None and abs(prompt_tps(fake) - 600) < 1e-6, "prompt_tps should be 600 for this example"
print("✅ Both formulas are correct.")

✅ Both formulas are correct.


## Step 2 — Experiments A and B
In each experiment **only one thing changes**:
- **A — quantization:** the same model in 3 precisions (q4_K_M = 4-bit, q8_0 = 8-bit, fp16 = 16-bit).
- **B — size:** the same model family in different sizes, all at q4_K_M.

The cell below picks the models for your track. Just run it.

In [9]:
print('Experiment A:',EXP_A)
print('Experiment B:',EXP_B)
print('Timed repetitions:',N_RUNS)
available={m['name'] for m in api('/api/tags')['models']}
for model in EXP_A+EXP_B:
    if model not in available:
        print('Downloading:',model,flush=True)
        pull(model)


Experiment A: ['llama3.2:1b-instruct-q4_K_M', 'llama3.2:1b-instruct-q8_0', 'llama3.2:1b-instruct-fp16']
Experiment B: ['qwen3:0.6b-q4_K_M', 'qwen3:1.7b-q4_K_M']
Timed repetitions: 3


### The measurement code (read it, then run it)
For each model, `profile_model` does:
1. a warm-up call (the first call also loads the model — we record the load time separately),
2. **checks that the model is 100% on the GPU** (if not, part of it runs on the slow CPU and the speed is not comparable),
3. `N_RUNS` timed answers to the same prompt,
4. a **quality check**: the model classifies the severity of 10 real incidents using the company's severity guide, and we compare with the gold answers,
5. saves one result row and unloads the model.

In [10]:
print('Fixed settings:',OPTIONS)
print('Standard prompt:',PROMPT)
print('Measurement phases: load, standard-prompt warm-up,',N_RUNS,'timed calls, 10 severity checks.')
print('Timing means and population SDs come from Ollama response fields, not token estimates.')


Fixed settings: {'temperature': 0, 'seed': 42, 'num_ctx': 2048, 'num_predict': 200, 'num_gpu': 0}
Standard prompt: Explain to a new on-call engineer, in at most 120 words, why a read replica can show higher latency than the primary database.
Measurement phases: load, standard-prompt warm-up, 3 timed calls, 10 severity checks.
Timing means and population SDs come from Ollama response fields, not token estimates.


### Run Experiment A (≈ 3 min on a T4)

In [11]:
for model in EXP_A:
    profile_model(model,'A')


### Run Experiment B (≈ 2–3 min on a T4)

In [12]:
for model in EXP_B:
    profile_model(model,'B')


### Your result tables

In [13]:
rows=read_jsonl(RESULTS/'profiles.jsonl')
columns=['model','processor','size_gb','vram_gb','nvidia_vram_delta_gb','decode_tps_mean','decode_tps_sd','prompt_tps_mean','prompt_tps_sd','warm_ttft_s_mean','warm_ttft_s_sd','load_s','severity_correct_of_10']
df=pd.DataFrame(rows)
for experiment in ['A','B']:
    print('Experiment',experiment)
    display(df[df.experiment==experiment][columns].round(3))
print('Model memory is from /api/ps; NVIDIA memory delta is measured before/after loading on GPU track.')
if TRACK=='GPU':
    bad=df[~df.gpu_comparable]
    print('Partially offloaded rows, excluded from GPU model selection:',list(bad.model))


Experiment A


,model,processor,size_gb,vram_gb,nvidia_vram_delta_gb,decode_tps_mean,decode_tps_sd,prompt_tps_mean,prompt_tps_sd,warm_ttft_s_mean,warm_ttft_s_sd,load_s,severity_correct_of_10
0,llama3.2:1b-instruct-q4_K_M,100% CPU,0.934,0.0,None,186.297,2.474,10550.399,523.352,0.005,0.000,8.256,5
3,llama3.2:1b-instruct-q8_0,100% CPU,1.447,0.0,None,110.326,2.917,4867.562,315.551,0.011,0.001,15.974,5
4,llama3.2:1b-instruct-fp16,100% CPU,2.606,0.0,None,60.527,1.344,2950.460,597.417,0.019,0.004,1.039,5


Experiment B


,model,processor,size_gb,vram_gb,nvidia_vram_delta_gb,decode_tps_mean,decode_tps_sd,prompt_tps_mean,prompt_tps_sd,warm_ttft_s_mean,warm_ttft_s_sd,load_s,severity_correct_of_10
1,qwen3:0.6b-q4_K_M,100% CPU,1.018,0.0,None,195.042,69.574,9522.524,1952.158,0.005,0.001,0.842,8
2,qwen3:1.7b-q4_K_M,100% CPU,1.878,0.0,None,126.484,5.652,5815.035,242.021,0.008,0.000,0.472,10


Model memory is from /api/ps; NVIDIA memory delta is measured before/after loading on GPU track.


### ✏️ YOUR TURN 2 — read your tables (write 2–3 sentences each)
1. **Experiment A:** how much memory (`size_gb`) and speed (`decode_tps_mean`) did q4_K_M save compared with fp16? Did `severity_correct_of_10` change?
2. **Experiment B:** where does accuracy stop improving enough to justify the slower speed?

**Answer 1.** In Experiment A, Q4 used 0.93 GB versus 2.61 GB for FP16, about 64% less loaded memory. It decoded 186.30 tokens/s compared with 60.53 tokens/s, a 3.08-fold difference. Q8 decoded 110.33 tokens/s. All three precisions scored 5/10 on severity, so the extra memory did not improve accuracy in this sample. Both tables report the mean and population standard deviation from Ollama timing fields, excluding the separate load and warm-up calls.

**Answer 2.** In Experiment B, qwen3:0.6b-q4_K_M scored 8/10 at 195.04 tokens/s, while qwen3:1.7b-q4_K_M scored 10/10 at 126.48 tokens/s. Loaded memory increased from 1.02 to 1.88 GB. I selected the larger model because its two additional correct classifications justified the lower measured speed for the later extraction tasks.

## Step 3 - Parallel requests
The dedicated server uses four GPU-track slots or two CPU-track slots. I do not restart another user service. I load the model before starting the wall-clock measurement, then compare one request with the track-specific simultaneous batch.

### ✏️ YOUR TURN 3 — choose the model for this test
Use the model you would pick from Step 2 (copy its name from the table).

In [14]:
chosen=choose_model(read_jsonl(RESULTS/'profiles.jsonl'))
PARALLEL_MODEL=chosen['model']
print('Parallel test model:',PARALLEL_MODEL)


Parallel test model: qwen3:1.7b-q4_K_M


In [15]:
unload_all()
par=[parallel_test(PARALLEL_MODEL,n) for n in [1,N_PARALLEL]]
display(pd.DataFrame([{k:v for k,v in row.items() if k not in ['memory','responses']} for row in par]))
for row in par:print(row['parallel_requests'],'requests: model memory GB',row['memory']['size_gb'],'model GPU GB',row['memory']['vram_gb'],'NVIDIA used GB',row['nvidia_memory_gb'])
if not (RESULTS/'model_card.json').exists():
    save_summary()


,model,parallel_requests,per_request_tps,per_request_tps_mean,aggregate_tps,wall_s,tokens_total,nvidia_memory_gb
0,qwen3:1.7b-q4_K_M,1,[119.0625304608168],119.062530,76.410964,0.745966,57,None
1,qwen3:1.7b-q4_K_M,2,"[67.15081078575713, 55.79800062912769]",61.474406,88.731372,1.465096,130,None


1 requests: model memory GB 1.87761664 model GPU GB 0.0 NVIDIA used GB None
2 requests: model memory GB 1.87761664 model GPU GB 0.0 NVIDIA used GB None


## Step 4 — Analysis and model card
### ✏️ YOUR TURN 4 — answer in your own words (≈ 250 words in total)
3. **Serving:** using the `aggregate_tps` from Step 3, estimate how many users who each send one request every 20 seconds (answers of ~200 tokens) this machine could serve within a 5-second target. Write your calculation and assumptions.
4. **Your choice:** which model will you use in the next practices, and why (speed, memory, accuracy)?

**Answer 3.** The two-request batch generated 130 tokens in 1.47 seconds, giving 88.73 tokens/s aggregate. Assuming 200 tokens per answer and one query every 20 seconds, a throughput-only projection is 88.73 x 20 / 200 = 8.9 users before headroom. The observed replies averaged 65 tokens. Their latency does not verify a five-second target for 200-token replies. I would test that output length and realistic arrivals before claiming capacity.

**Answer 4.** I selected qwen3:1.7b-q4_K_M for subsequent work and recorded its measured speed, memory and 10/10 severity score in the model card. These are local CPU results on Apple M4 Pro, with zero model GPU memory. I kept temperature 0, seed 42, context 2048 and a 200-token generation limit. Each speed estimate uses three timed runs. The small ten-incident quality sample limits generalisation to other workloads.

### ✏️ YOUR TURN 5 — your model card
A **model card** is a short record of the model you choose for the rest of the course and *why*. In Lab 2 and later you will use `MODEL_CARD["model"]` as your model.

| Field | What to write | Where to find it |
|---|---|---|
| `model` | the exact model name you choose | column `model` in your Step 2 tables |
| `decode_tps` | its speed (tokens per second) | column `decode_tps_mean` |
| `size_gb` | its size in memory | column `size_gb` |
| `severity_correct_of_10` | its quality score | column `severity_correct_of_10` |
| `reason_for_choice` | **one sentence**: why this model is the best trade-off for you | your answers in YOUR TURN 2 and 4 |

Example: `"reason_for_choice": "Fast (60 tok/s), fits easily in memory and scored 8/10, as good as the larger model."`

In [16]:
MODEL_CARD=json.loads((RESULTS/'model_card.json').read_text())
assert MODEL_CARD['model'] and MODEL_CARD['reason_for_choice']
assert MODEL_CARD['model']==PARALLEL_MODEL, 'Model card differs from the measured model selection.'
display(MODEL_CARD)


{'model': 'qwen3:1.7b-q4_K_M',
 'track': 'CPU',
 'runtime': 'local CPU, Apple M4 Pro',
 'decode_tps': 126.48373503343504,
 'size_gb': 1.87761664,
 'vram_gb': 0.0,
 'severity_correct_of_10': 10,
 'reason_for_choice': 'I selected highest measured severity accuracy among Qwen size candidates, breaking ties by measured speed and memory efficiency; accuracy 10/10, speed 126.48 tok/s, memory 1.88 GB.'}

## Step 5 — Same call through LangChain
In the next labs we use LangChain, which hides the provider behind one interface. Run the cell: it sends the Step 2 prompt to your model through LangChain.

In [17]:
langchain_path=RESULTS/'langchain_response.json'
if langchain_path.exists():
    lc_result=json.loads(langchain_path.read_text())
else:
    lc_result=langchain_call(MODEL_CARD['model'])
print('Actual LangChain response:')
print(lc_result['content'])
print('Measured wall seconds:',lc_result['wall_s'])


Actual LangChain response:
A read replica can have higher latency because it is a separate database that handles read requests, leading to additional network overhead and potential delays in data synchronization. The primary database is optimized for write operations, while the replica is designed for read, which can result in slower response times due to less optimization and higher network latency.
Measured wall seconds: 0.6537722499997471


## Optional — what CPU offload looks like (GPU track only)
We force Ollama to keep only 10 layers on the GPU. Compare `gpu_share` and `decode_tps_mean` with the normal row, and explain in two sentences what happened.

In [18]:
print('The forced CPU-offload demonstration is optional and was not included in this run.')


The forced CPU-offload demonstration is optional and was not included in this run.


The optional forced-CPU-offload demonstration was omitted; the required experiments above record the real processor split for each model.

## Submission
This notebook contains CPU-track measurements from local CPU, Apple M4 Pro, both experiment tables, parallel serving results, analysis, the selected model card and a real LangChain response. The accompanying results folder preserves complete responses and timings. Download the executed notebook and submit it to Moodle.